In [4]:
3

3

In [6]:
import pymupdf  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
print("30%")
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
print("60%")
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
print("80%")
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS



30%
60%
80%


In [7]:
###Clip Model
import os
from dotenv import load_dotenv
load_dotenv()

## set up the environment
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("done")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

done


In [16]:
def embed_image(image_data):
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        # Extract tensor if output is a model dataclass (BaseModelOutputWithPooling)
        if hasattr(features, "pooler_output") and features.pooler_output is not None:
            features = features.pooler_output
        elif hasattr(features, "image_embeds") and features.image_embeds is not None:
            features = features.image_embeds
        elif isinstance(features, (tuple, list)):
            features = features[0]

        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()
def embed_text(text):
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        
        # 🔹 Extract tensor if output is a model dataclass (BaseModelOutputWithPooling)
        if hasattr(features, "pooler_output") and features.pooler_output is not None:
            features = features.pooler_output
        elif hasattr(features, "text_embeds") and features.text_embeds is not None:
            features = features.text_embeds
        elif isinstance(features, (tuple, list)):
            features = features[0]

        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()


def embed_image(image_data):
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        
        # 🔹 Extract tensor if output is a model dataclass (BaseModelOutputWithPooling)
        if hasattr(features, "pooler_output") and features.pooler_output is not None:
            features = features.pooler_output
        elif hasattr(features, "image_embeds") and features.image_embeds is not None:
            features = features.image_embeds
        elif isinstance(features, (tuple, list)):
            features = features[0]

        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().numpy()



In [17]:
## Process PDF
pdf_path="07_multimodle RAG\multimodal_sample.pdf"
doc=pymupdf.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

# Text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)




In [18]:
doc

Document('07_multimodle RAG\multimodal_sample.pdf')

In [19]:
# Iterate through each page of the opened PDF document
for i, page in enumerate(doc):
    
    # ==========================================
    # 1. PROCESS TEXT CONTENT
    # ==========================================
    text = page.get_text()
    
    # Only process non-empty text pages (ignore blank spaces/newlines)
    if text.strip():
        # Create Document objects and split into chunks in a single step,
        # attaching page metadata (page number 'i' and type 'text') to each chunk
        text_chunks = splitter.create_documents(
            texts=[text], 
            metadatas=[{"page": i, "type": "text"}]
        )
        
        # Add the chunk Documents to our master document list
        all_docs.extend(text_chunks)
        
        # Generate CLIP text embeddings for each chunk and append to master embedding list
        all_embeddings.extend([embed_text(chunk.page_content) for chunk in text_chunks])

    # ==========================================
    # 2. PROCESS IMAGE ELEMENTS
    # ==========================================
    # Extract all embedded images on the current page
    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            # Step 2a: Get PDF object reference ID (xref) for the image
            xref = img[0]
            
            # Step 2b: Extract raw binary image data from the PDF stream
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]  # Raw bytes (PNG/JPEG)
            
            # Step 2c: Generate a unique ID for this image (e.g., 'page_0_img_0')
            image_id = f"page_{i}_img_{img_index}"
            
            # Step 2d: Encode raw binary image bytes into a base64 string for Vision LLM (GPT-4o)
            image_data_store[image_id] = base64.b64encode(image_bytes).decode()
            
            # Step 2e: Convert raw bytes to a PIL RGB image for neural network processing
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Step 2f: Generate CLIP visual embedding vector for cross-modal vector search
            all_embeddings.append(embed_image(pil_image))
            
            # Step 2g: Create a placeholder Document carrying image metadata & lookup ID
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            # Handle corrupted or unparseable image objects gracefully
            print(f"Error processing image {img_index} on page {i}: {e}")

# Close the PDF document to release file system handles and memory
doc.close()


In [20]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.00267244,  0.01283   , -0.05183139, ..., -0.00385087,
         0.02977722, -0.00010683],
       [ 0.01732337, -0.0132769 , -0.0242703 , ...,  0.08994054,
        -0.00272156,  0.03253038]], shape=(2, 512), dtype=float32)